In [16]:
import os
os.environ["HF_HOME"] = "/shared/data3/pk36/.cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys

# Get the parent directory's path
parent_dir = os.path.dirname(os.getcwd())

# Add the parent directory to the system path
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [51]:
import glob
import json
import ast
from collections import defaultdict

In [18]:
directory_path = "../inspiration_pred_output/"
search_pattern = os.path.join(directory_path, '*.json')
json_files = glob.glob(search_pattern)

In [52]:
with open("../evaluation/processed_abstracts.json", "r") as f:
    groundtruth = json.load(f)

In [53]:
all_ids = {"_".join(key.split('_', maxsplit=2)[:2]):key for key in groundtruth.keys()}

In [54]:
len(all_ids)

1521

In [60]:
list(mainmethod_results.keys())

['174_34799_few-shot_learning',
 '32596_31621_the_identification_of_blackbox_optimization_landscape_features',
 '1799_11198_video_synthesis',
 '3588_7784_the_augmentation_policy',
 '31012_8384_skin_attributes_detection',
 '7621_9828_the_problem_of_coupling_vision-based_navigation_systems_for_unmanned_aerial_vehicles_with_robust_obstacle_avoidance_capabilities',
 '19225_9440_apply_deep_learning/machine_learning_methods_to_disease_classification_tasks',
 '20712_18723_unsupervised_learning_of_intermediate_representations_utilizing_abundant_unlabeled_sensory_data',
 '23785_39946_learning_representations_from_set-structured_data',
 '1432_24504_ai_systems',
 '1367_25555_counting',
 '29558_34759_a_feature_transform_technique_that_imposes_invariance_properties_in_the_training_of_deep_neural_networks',
 '21031_874_action_segmentation',
 '7245_11725_low_cost_robots,_such_as_vacuum_cleaners_or_lawn_mowers,_employ_simplistic_and_often_random_navigation_policies',
 '16375_11296_a_stylized_model_of_

### Ablation

In [55]:
directory_path = "../evaluation/ablations/no_decomp_outputs"
search_pattern = os.path.join(directory_path, '*.json')
ablation_json_files = glob.glob(search_pattern)
print(ablation_json_files)

['../evaluation/ablations/no_decomp_outputs/174_34799_few-shot_learning_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/28782_1293_the_identity_matchi_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/35073_13117_the_algorithmic_de_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/4070_23048_the_path_planning_p_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/1799_11198_video_synthesis_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/3588_7784_the_augmentation_pol_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/36120_13130_the_shape_generati_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/19225_9440_apply_deep_learning_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/20712_18723_unsupervised_learn_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/23785_39946_learning_represent_20_predictions.json', '../evaluation/ablations/no_decomp_outputs/143

In [61]:
formatted_outputs = {}
invalid_files = []
count_valid, count_invalid = 0, 0
# Iterate through the list of file paths
for idx, file_path in enumerate(json_files):
    id = "_".join(os.path.basename(file_path).split("_", maxsplit=2)[:2])
    search_pattern = os.path.join("../evaluation/ablations/no_decomp_outputs", f'{id}*.json')
    ablation_json_file = glob.glob(search_pattern)
    if len(ablation_json_file):
        ablation_json_file = ablation_json_file[0]
    else:
        continue

    full_id = all_ids[id]
    if full_id not in mainmethod_results:
        continue
    # Open each file using a context manager
    with open(ablation_json_file, 'r', encoding='utf-8') as f:
        # Load the JSON data from the file
        data = json.load(f)
        # Check if valid output
        keys = [k for k in data.keys() if "idea_rankings" not in k]
        if (len(keys) > 4) and ("idea_rankings" in data) and (len(data["idea_rankings"]) >= 1):
            formatted_outputs[full_id] = {
                "research_problem": data["research_problem"],
                "target_domain": data["target_domain"],
                "target_domain_subfield": data["fine_grained_domain"],
                "predicted_takeaways": [{"source_domain": idea["source_domain"],
                                         "integration_mechanism": {
                                             "target_domain_elements": idea["idea_fragment"]["integration_mechanism"]["target_domain_elements"],
                                             "source_domain_takeaways": idea["idea_fragment"]["integration_mechanism"]["selected_takeaways"]
                                         }
                                         }
                                        for idea in data["idea_rankings"]],
                "gt_takeaways": {"source_domain":  groundtruth[full_id]["original_data"]["source_domain"],
                                 "integration_mechanism": groundtruth[full_id]["processed_abstract"]["integration_mechanism"]
                                 },
                "proposed_ideas": [{"source_domain": idea["source_domain"],
                                    "idea": idea["idea_fragment"]["concrete_realization"]
                                    } for idea in data["idea_rankings"]],
                "gt_idea": {"source_domain":  groundtruth[full_id]["original_data"]["source_domain"],
                            "idea": groundtruth[full_id]["processed_abstract"]["concrete_realization"]
                }

            }
            count_valid += 1
        else:
            invalid_files.append(full_id)
            count_invalid +=1

In [62]:
len(formatted_outputs)

400

In [63]:
with open("final_nodecomp_results.json", "w") as f:
    json.dump(formatted_outputs, f, indent=2)

### Main Method

In [12]:
formatted_outputs = {}
invalid_files = []
count_valid, count_invalid = 0, 0
# Iterate through the list of file paths
for idx, file_path in enumerate(json_files):
    id = "_".join(os.path.basename(file_path).split("_", maxsplit=2)[:2])
    full_id = all_ids[id]
    # Open each file using a context manager
    with open(file_path, 'r', encoding='utf-8') as f:
        # Load the JSON data from the file
        data = json.load(f)
        # Check if valid output
        keys = [k for k in data.keys() if "idea_rankings" not in k]
        if (len(keys) > 4) and ("idea_rankings" in data) and (len(data["idea_rankings"]) >= 1):
            formatted_outputs[full_id] = {
                "research_problem": data["research_problem"],
                "target_domain": data["target_domain"],
                "target_domain_subfield": data["fine_grained_domain"],
                "predicted_takeaways": [{"source_domain": idea["source_domain"],
                                         "integration_mechanism": {
                                             "target_domain_elements": idea["idea_fragment"]["integration_mechanism"]["target_domain_elements"],
                                             "source_domain_takeaways": idea["idea_fragment"]["integration_mechanism"]["selected_takeaways"]
                                         }
                                         }
                                        for idea in data["idea_rankings"]],
                "gt_takeaways": {"source_domain":  groundtruth[full_id]["original_data"]["source_domain"],
                                 "integration_mechanism": groundtruth[full_id]["processed_abstract"]["integration_mechanism"]
                                 },
                "proposed_ideas": [{"source_domain": idea["source_domain"],
                                    "idea": idea["idea_fragment"]["concrete_realization"]
                                    } for idea in data["idea_rankings"]],
                "gt_idea": {"source_domain":  groundtruth[full_id]["original_data"]["source_domain"],
                            "idea": groundtruth[full_id]["processed_abstract"]["concrete_realization"]
                }

            }
            count_valid += 1
        else:
            invalid_files.append(full_id)
            count_invalid +=1

In [13]:
with open("final_mainmethod_results.json", "w") as f:
    json.dump(formatted_outputs, f, indent=2)

In [14]:
len(formatted_outputs)

400

### Baseline #1

In [21]:
with open("../evaluation/baseline_output/baseline1_direct.json", "r") as f:
    baseline_one = json.load(f)

In [22]:
formatted_baseone_outputs = {}
for id in formatted_outputs:
    baseline_data = baseline_one[id]
    formatted_baseone_outputs[id] = {
                "research_problem": baseline_data["research_problem"],
                "target_domain": formatted_outputs[id]["target_domain"],
                "target_domain_subfield": formatted_outputs[id]["target_domain_subfield"],
                "all_subfields": [idea["fine_grained_source_domain"] for idea in baseline_data["idea_rankings"]],
                "predicted_takeaways": [{"source_domain": idea["source_domain"],
                                         "integration_mechanism": {
                                             "target_domain_elements": idea["idea_fragment"]["integration_mechanism"]["target_domain_elements"],
                                             "source_domain_takeaways": idea["idea_fragment"]["integration_mechanism"]["source_domain_takeaways"]
                                         }
                                         }
                                        for idea in baseline_data["idea_rankings"]],
                "gt_takeaways": {"source_domain":  groundtruth[id]["original_data"]["source_domain"],
                                 "integration_mechanism": groundtruth[id]["processed_abstract"]["integration_mechanism"]
                                 },
                "proposed_ideas": [{"source_domain": idea["source_domain"],
                                    "idea": idea["idea_fragment"]["concrete_realization"]
                                    } for idea in baseline_data["idea_rankings"]],
                "gt_idea": {"source_domain":  groundtruth[id]["original_data"]["source_domain"],
                            "idea": groundtruth[id]["processed_abstract"]["concrete_realization"]
                }

            }

In [23]:
len(formatted_baseone_outputs)

400

In [24]:
with open("final_baseline_one_results.json", "w") as f:
    json.dump(formatted_baseone_outputs, f, indent=2)

### Baseline #2

In [25]:
with open("../evaluation/baseline_output/baseline2_dual_retrieval.json", "r") as f:
    baseline_two = json.load(f)

In [ ]:
formatted_basetwo_outputs = {}
for id in formatted_outputs:
    baseline_data = baseline_two[id]
    formatted_basetwo_outputs[id] = {
                "research_problem": baseline_data["research_problem"],
                "target_domain": formatted_outputs[id]["target_domain"],
                "target_domain_subfield": formatted_outputs[id]["target_domain_subfield"],
                "predicted_takeaways": [{"source_domain": idea["source_domain"],
                                         "integration_mechanism": {
                                             "target_domain_elements": idea["idea_fragment"]["integration_mechanism"]["target_domain_elements"],
                                             "source_domain_takeaways": idea["idea_fragment"]["integration_mechanism"]["source_domain_takeaways"]
                                         }
                                         }
                                        for idea in baseline_data["idea_rankings"]],
                "gt_takeaways": {"source_domain":  groundtruth[id]["original_data"]["source_domain"],
                                 "integration_mechanism": groundtruth[id]["processed_abstract"]["integration_mechanism"]
                                 },
                "proposed_ideas": [{"source_domain": idea["source_domain"],
                                    "idea": idea["idea_fragment"]["concrete_realization"]
                                    } for idea in baseline_data["idea_rankings"]],
                "gt_idea": {"source_domain":  groundtruth[id]["original_data"]["source_domain"],
                            "idea": groundtruth[id]["processed_abstract"]["concrete_realization"]
                }

            }

In [27]:
len(formatted_basetwo_outputs)

400

In [28]:
with open("final_baseline_two_results.json", "w") as f:
    json.dump(formatted_basetwo_outputs, f, indent=2)

## Results Analysis

In [12]:
def win_rate_computation(method_refs, k):
    for eval_type, eval_results in method_refs.items():
        print(f"{eval_type} Evaluation:")
        for method_name, method_vals in eval_results.items():
            print(f"\t{method_name}:")
            count = 0
            stats = defaultdict(lambda: {"wins": 0, "ties": 0, "losses": 0})
            for key, value in method_vals.items():
                if key == "final_stats":
                    continue
                if type(value) != dict:
                    continue

                sample_id, idx = ast.literal_eval(key)
                # if idx >= k:
                if (idx >= k) or str((sample_id, k-1)) not in method_refs[eval_type]["Main Method"]:
                    continue
                else:
                    count += 1


                for criteria in value[f"{eval_type}_comparison"].keys():
                    if value[f"{eval_type}_comparison"][criteria]['preferred_method'] == 1:
                        stats[criteria]["wins"] += 1
                    elif value[f"{eval_type}_comparison"][criteria]['preferred_method'] == 2:
                        stats[criteria]["losses"] += 1
                    else:
                        print(f"invalid {sample_id}")
                if value["overall_assessment"]["preferred_method"] == 1:
                    stats["overall"]["wins"] += 1
                elif value["overall_assessment"]["preferred_method"] == 2:
                    stats["overall"]["losses"] += 1
                else:
                    print(f"invalid {sample_id}")


            for key, value in stats.items():
                total = value["wins"] + value["ties"] + value["losses"]
                win_rate = value["wins"] / total if total > 0 else 0
                loss_rate = value["losses"] / total if total > 0 else 0
                print(f"\t\t{key} ({count}): Win Rate @ {k}: {win_rate:.2f}, Loss Rate: {loss_rate:.2f}")

In [4]:
with open("../evaluation/comparison/run_3/mainmethod_idea_eval.json") as f:
    mainmethod_idea_results = json.load(f)

with open("../evaluation/comparison/run_3/mainmethod_takeaway_eval.json") as f:
    mainmethod_takeaway_results = json.load(f)

with open("../evaluation/comparison/run_3/baseline_one_idea_eval.json") as f:
    baseline_one_idea_results = json.load(f)

with open("../evaluation/comparison/run_3/baseline_one_takeaway_eval.json") as f:
    baseline_one_takeaway_results = json.load(f)

with open("../evaluation/comparison/run_3/baseline_two_idea_eval.json") as f:
    baseline_two_idea_results = json.load(f)

with open("../evaluation/comparison/run_3/baseline_two_takeaway_eval.json") as f:
    baseline_two_takeaway_results = json.load(f)

In [34]:
with open("final_mainmethod_results.json", "r") as f:
    mainmethod_results = json.load(f)

with open("final_baseline_one_results.json", "r") as f:
    b_one_results = json.load(f)

with open("final_baseline_two_results.json", "r") as f:
    b_two_results = json.load(f)

In [56]:
for key, value in mainmethod_idea_results.items():
    if key == "final_stats":
        continue
    if type(value) != dict:
        continue

    sample_id, idx = ast.literal_eval(key)
    if idx >= 1:
        continue
    if (baseline_two_idea_results[key]["idea_comparison"]["usefulness"]["preferred_method"] == 1) and (value["idea_comparison"]["usefulness"]["preferred_method"] == 2):
        main_source = mainmethod_results[sample_id]["proposed_ideas"][idx]["source_domain"]
        btwo_source = b_two_results[sample_id]["proposed_ideas"][idx]["source_domain"]
        if main_source == btwo_source:
            print(sample_id, idx, main_source, btwo_source)

1799_11198_video_synthesis 0 Engineering Engineering
37209_33556_improve_models'_robustness_to_distributional_shifts 0 Psychology Psychology
489_29921_the_decision-making_problem 0 Psychology Psychology
33860_19347_individuals_who_use_myoelectric_upper-limb_prostheses 0 Psychology Psychology
4149_3861_machine_learning_models 0 Psychology Psychology
3708_5072_one-shot_detection 0 Psychology Psychology
8699_10270_applying_convolutional_neural_networks_to_spherical_images 0 Mathematics Mathematics
9795_1221_the_iris_network_design_process 0 Engineering Engineering
10162_1174_discover_the_common_and_salient_objects_from_a_group_of_relevant_images 0 Psychology Psychology
7485_328_learning_and_recognizing_objects_from_few_images 0 Psychology Psychology
32468_7482_token-to-expert_allocation 0 Operations Research Operations Research
3205_1594_object_detection 0 Psychology Psychology
17378_795_train_interpretable_convolutional_neural_networks 0 Physics Physics
15582_17524_both_image_embeddings_

In [5]:
# method_refs = {"Main Method": {"ideas": mainmethod_idea_results, "takeaway": mainmethod_takeaway_results},
#                "Baseline 1": {"ideas": baseline_one_idea_results, "takeaway": baseline_one_takeaway_results},
#                "Baseline 2": {"ideas": baseline_two_idea_results, "takeaway": baseline_two_takeaway_results}}

method_refs = {"idea": {"Main Method": mainmethod_idea_results,
                         "Baseline 1": baseline_one_idea_results,
                         "Baseline 2": baseline_two_idea_results
                         },
                "takeaway": {"Main Method": mainmethod_takeaway_results,
                             "Baseline 1": baseline_one_takeaway_results,
                             "Baseline 2": baseline_two_takeaway_results}}

In [13]:
# WITH TIE
win_rate_computation(method_refs=method_refs, k=1)

idea Evaluation:
	Main Method:
		interdisciplinary_novelty (400): Win Rate @ 1: 0.83, Loss Rate: 0.17
		interdisciplinary_usefulness (400): Win Rate @ 1: 0.66, Loss Rate: 0.34
		overall (400): Win Rate @ 1: 0.71, Loss Rate: 0.29
	Baseline 1:
		interdisciplinary_novelty (400): Win Rate @ 1: 0.14, Loss Rate: 0.86
		interdisciplinary_usefulness (400): Win Rate @ 1: 0.35, Loss Rate: 0.65
		overall (400): Win Rate @ 1: 0.30, Loss Rate: 0.69
	Baseline 2:
		interdisciplinary_novelty (400): Win Rate @ 1: 0.68, Loss Rate: 0.32
		interdisciplinary_usefulness (400): Win Rate @ 1: 0.70, Loss Rate: 0.30
		overall (400): Win Rate @ 1: 0.72, Loss Rate: 0.28
takeaway Evaluation:
	Main Method:
		interdisciplinary_insightfulness (400): Win Rate @ 1: 0.85, Loss Rate: 0.14
		interdisciplinary_relevance (400): Win Rate @ 1: 0.60, Loss Rate: 0.40
		overall (400): Win Rate @ 1: 0.64, Loss Rate: 0.36
	Baseline 1:
		interdisciplinary_insightfulness (400): Win Rate @ 1: 0.18, Loss Rate: 0.82
		interdisciplinary

In [14]:
# WITH TIE
win_rate_computation(method_refs=method_refs, k=2)

idea Evaluation:
	Main Method:
		interdisciplinary_novelty (620): Win Rate @ 2: 0.84, Loss Rate: 0.16
		interdisciplinary_usefulness (620): Win Rate @ 2: 0.66, Loss Rate: 0.34
		overall (620): Win Rate @ 2: 0.71, Loss Rate: 0.29
	Baseline 1:
		interdisciplinary_novelty (620): Win Rate @ 2: 0.17, Loss Rate: 0.83
		interdisciplinary_usefulness (620): Win Rate @ 2: 0.40, Loss Rate: 0.60
		overall (620): Win Rate @ 2: 0.35, Loss Rate: 0.65
	Baseline 2:
		interdisciplinary_novelty (620): Win Rate @ 2: 0.70, Loss Rate: 0.30
		interdisciplinary_usefulness (620): Win Rate @ 2: 0.65, Loss Rate: 0.35
		overall (620): Win Rate @ 2: 0.68, Loss Rate: 0.32
takeaway Evaluation:
	Main Method:
		interdisciplinary_insightfulness (620): Win Rate @ 2: 0.85, Loss Rate: 0.15
		interdisciplinary_relevance (620): Win Rate @ 2: 0.62, Loss Rate: 0.38
		overall (620): Win Rate @ 2: 0.66, Loss Rate: 0.34
	Baseline 1:
		interdisciplinary_insightfulness (620): Win Rate @ 2: 0.23, Loss Rate: 0.77
		interdisciplinary

In [15]:
# NO TIE
win_rate_computation(method_refs=method_refs, k=3)

idea Evaluation:
	Main Method:
		interdisciplinary_novelty (702): Win Rate @ 3: 0.83, Loss Rate: 0.17
		interdisciplinary_usefulness (702): Win Rate @ 3: 0.66, Loss Rate: 0.34
		overall (702): Win Rate @ 3: 0.70, Loss Rate: 0.30
	Baseline 1:
		interdisciplinary_novelty (702): Win Rate @ 3: 0.20, Loss Rate: 0.80
		interdisciplinary_usefulness (702): Win Rate @ 3: 0.40, Loss Rate: 0.60
		overall (702): Win Rate @ 3: 0.35, Loss Rate: 0.65
	Baseline 2:
		interdisciplinary_novelty (702): Win Rate @ 3: 0.68, Loss Rate: 0.32
		interdisciplinary_usefulness (702): Win Rate @ 3: 0.64, Loss Rate: 0.36
		overall (702): Win Rate @ 3: 0.66, Loss Rate: 0.34
takeaway Evaluation:
	Main Method:
		interdisciplinary_insightfulness (702): Win Rate @ 3: 0.84, Loss Rate: 0.16
		interdisciplinary_relevance (702): Win Rate @ 3: 0.61, Loss Rate: 0.39
		overall (702): Win Rate @ 3: 0.66, Loss Rate: 0.34
	Baseline 1:
		interdisciplinary_insightfulness (702): Win Rate @ 3: 0.27, Loss Rate: 0.73
		interdisciplinary

### Direct Comparison

In [ ]:
import json
import glob
import os
from collections import defaultdict

def load_json(path):
    with open(path) as f:
        return json.load(f)

def compute_winrates(eval_dict, comparison_key, top_k=1):
    """
    eval_dict: loaded JSON
    comparison_key: 'idea_comparison' or 'takeaway_comparison'
    """
    stats = defaultdict(lambda: {"wins": 0, "losses": 0, "ties": 0})

    for k, v in eval_dict.items():
        if type(v) == list:
            print(k, comparison_key)
        if k == "final_stats":
            continue

        sample_id, idx = ast.literal_eval(k)
        if int(idx) >= top_k:
            continue
        
        comp = v.get(comparison_key, {})
        for criterion, result in comp.items():
            pref = result.get("preferred_method")
            if pref == 1:
                stats[criterion]["wins"] += 1
            elif pref == 2:
                stats[criterion]["losses"] += 1

    return stats

def print_stats(pair_id, level, stats):
    print(f"\n=== {pair_id} | {level.upper()} ===")
    for criterion, s in stats.items():
        total = s["wins"] + s["losses"] + s["ties"]
        if total == 0:
            continue
        win_rate = s["wins"] / total
        loss_rate = s["losses"] / total
        tie_rate = s["ties"] / total

        print(
            f"{criterion}: "
            f"win={win_rate:.2%}, "
            f"loss={loss_rate:.2%}, "
            f"tie={tie_rate:.2%} "
            f"(n={total})"
        )

In [132]:
def main(eval_dir, top_k=1):
    idea_files = glob.glob(os.path.join(eval_dir, "*_idea_eval.json"))
    takeaway_files = glob.glob(os.path.join(eval_dir, "*_takeaway_eval.json"))

    for path in sorted(idea_files):
        pair_id = os.path.basename(path).replace("_idea_eval.json", "")
        data = load_json(path)
        stats = compute_winrates(data, "idea_comparison", top_k=top_k)
        print_stats(pair_id, "idea", stats)

    for path in sorted(takeaway_files):
        pair_id = os.path.basename(path).replace("_takeaway_eval.json", "")
        data = load_json(path)
        stats = compute_winrates(data, "takeaway_comparison", top_k=top_k)
        print_stats(pair_id, "takeaway", stats)

In [133]:
main(eval_dir="../evaluation/direct_comparison", top_k=1)


=== baseline_one_vs_baseline_two | IDEA ===
interdisciplinary_novelty: win=10.00%, loss=90.00%, tie=0.00% (n=400)
interdisciplinary_usefulness: win=23.00%, loss=77.00%, tie=0.00% (n=400)

=== baseline_one_vs_mainmethod | IDEA ===
interdisciplinary_novelty: win=5.50%, loss=94.50%, tie=0.00% (n=400)
interdisciplinary_usefulness: win=34.00%, loss=66.00%, tie=0.00% (n=400)

=== baseline_two_vs_mainmethod | IDEA ===
interdisciplinary_novelty: win=33.25%, loss=66.75%, tie=0.00% (n=400)
interdisciplinary_usefulness: win=65.50%, loss=34.50%, tie=0.00% (n=400)

=== baseline_one_vs_baseline_two | TAKEAWAY ===
interdisciplinary_insightfulness: win=13.25%, loss=86.75%, tie=0.00% (n=400)
interdisciplinary_relevance: win=44.75%, loss=55.25%, tie=0.00% (n=400)

=== baseline_one_vs_mainmethod | TAKEAWAY ===
interdisciplinary_insightfulness: win=8.75%, loss=91.25%, tie=0.00% (n=400)
interdisciplinary_relevance: win=53.50%, loss=46.50%, tie=0.00% (n=400)

=== baseline_two_vs_mainmethod | TAKEAWAY ===
i

In [134]:
main(eval_dir="../evaluation/direct_comparison", top_k=2)


=== baseline_one_vs_baseline_two | IDEA ===
interdisciplinary_novelty: win=11.00%, loss=89.00%, tie=0.00% (n=800)
interdisciplinary_usefulness: win=31.00%, loss=69.00%, tie=0.00% (n=800)

=== baseline_one_vs_mainmethod | IDEA ===
interdisciplinary_novelty: win=5.49%, loss=94.51%, tie=0.00% (n=710)
interdisciplinary_usefulness: win=39.15%, loss=60.85%, tie=0.00% (n=710)

=== baseline_two_vs_mainmethod | IDEA ===
interdisciplinary_novelty: win=36.06%, loss=63.94%, tie=0.00% (n=710)
interdisciplinary_usefulness: win=62.54%, loss=37.46%, tie=0.00% (n=710)

=== baseline_one_vs_baseline_two | TAKEAWAY ===
interdisciplinary_insightfulness: win=15.88%, loss=84.12%, tie=0.00% (n=800)
interdisciplinary_relevance: win=49.00%, loss=51.00%, tie=0.00% (n=800)

=== baseline_one_vs_mainmethod | TAKEAWAY ===
interdisciplinary_insightfulness: win=9.86%, loss=90.14%, tie=0.00% (n=710)
interdisciplinary_relevance: win=54.37%, loss=45.63%, tie=0.00% (n=710)

=== baseline_two_vs_mainmethod | TAKEAWAY ===
i

In [135]:
main(eval_dir="../evaluation/direct_comparison", top_k=3)


=== baseline_one_vs_baseline_two | IDEA ===
interdisciplinary_novelty: win=11.75%, loss=88.25%, tie=0.00% (n=1200)
interdisciplinary_usefulness: win=33.25%, loss=66.75%, tie=0.00% (n=1200)

=== baseline_one_vs_mainmethod | IDEA ===
interdisciplinary_novelty: win=6.14%, loss=93.86%, tie=0.00% (n=944)
interdisciplinary_usefulness: win=41.74%, loss=58.26%, tie=0.00% (n=944)

=== baseline_two_vs_mainmethod | IDEA ===
interdisciplinary_novelty: win=37.92%, loss=62.08%, tie=0.00% (n=944)
interdisciplinary_usefulness: win=62.61%, loss=37.39%, tie=0.00% (n=944)

=== baseline_one_vs_baseline_two | TAKEAWAY ===
interdisciplinary_insightfulness: win=17.67%, loss=82.33%, tie=0.00% (n=1200)
interdisciplinary_relevance: win=51.17%, loss=48.83%, tie=0.00% (n=1200)

=== baseline_one_vs_mainmethod | TAKEAWAY ===
interdisciplinary_insightfulness: win=11.55%, loss=88.45%, tie=0.00% (n=944)
interdisciplinary_relevance: win=54.98%, loss=45.02%, tie=0.00% (n=944)

=== baseline_two_vs_mainmethod | TAKEAWAY 

### Source Diversity

In [9]:
from collections import Counter
import plotly.express as px
import pandas as pd
from pydantic import BaseModel

In [4]:
from utils import batch_llm_inference
from vllm import LLM

/home/pk36/structured_survey/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-31 15:14:39 [__init__.py:216] Automatically detected platform cuda.


In [ ]:
print("Loading model...")
llm = LLM(model="Qwen/Qwen3-14B", tensor_parallel_size=1)
print("Model loaded.\n")

In [6]:
def plot_target_dist(data, name="IdeaCatalyst"):
    # --- Collect source domains ---
    target_domains = []

    for sample in data.values():
        domain = sample.get("target_domain")
        target_domains.append(domain)

    # --- Count and sort domains ---
    domain_counts = Counter(target_domains)

    df = (
        pd.DataFrame(domain_counts.items(), columns=["Target Domain", "Count"])
        .sort_values("Count", ascending=False)
    )

    # --- Create bar plot ---
    fig = px.bar(
        df,
        x="Target Domain",
        y="Count",
        color="Target Domain",   # adds color variation
        text="Count",
        title=f"Distribution of Target Domains in {name}"
    )

    # --- Styling for modern look ---
    fig.update_traces(
        textposition="outside"
    )

    fig.update_layout(
        template="seaborn",
        height=600,              # increased height
        font=dict(size=14),
        title_font=dict(size=22),
        xaxis_title_font=dict(size=16),
        yaxis_title_font=dict(size=16),
        bargap=0.3,
        showlegend=False         # cleaner since color = domain
    )

    fig.update_yaxes(
        range=[0, df["Count"].max() * 1.15]  # prevents top cutoff
    )

    fig.show()

In [36]:
def plot_source_dist_multi(method_data_dict):
    """
    method_data_dict: dict
        key   -> method name (str)
        value -> data in the same format as before (dict or list)
    """

    records = []

    for method_name, data in method_data_dict.items():
        source_domains = []

        if type(data) != list:
            for sample in data.values():
                for idx, idea_entry in enumerate(sample.get("proposed_ideas", [])):
                    if idx >= 2:  # keep your top-2 constraint
                        break
                    domain = idea_entry.get("source_domain")
                    if domain is not None:
                        source_domains.append(domain)
        else:
            source_domains = data

        domain_counts = Counter(source_domains)

        for domain, count in domain_counts.items():
            if count < 10:
                continue
            records.append({
                "Method": method_name,
                "Source Domain": domain,
                "Count": count
            })

    # --- Create combined DataFrame ---
    df = pd.DataFrame(records)

    # Ensure consistent domain ordering (global sort)
    domain_order = (
        df.groupby("Source Domain")["Count"]
          .sum()
          .sort_values(ascending=False)
          .index
          .tolist()
    )

    # --- Plot grouped bar chart ---
    fig = px.bar(
        df,
        x="Source Domain",
        y="Count",
        color="Method",
        barmode="group",
        category_orders={"Source Domain": domain_order},
        title=f"Distribution of Source Domains Across Methods",
        text="Count"
    )

    # --- Styling ---
    fig.update_traces(
        textposition="outside"
    )

    fig.update_layout(
        template="seaborn",
        height=650,
        font=dict(size=14),
        title_font=dict(size=22),
        xaxis_title_font=dict(size=16),
        yaxis_title_font=dict(size=16),
        bargap=0.25,
        bargroupgap=0.1
    )

    fig.update_yaxes(
        range=[0, df["Count"].max() * 1.2]
    )

    fig.show()

In [29]:
def plot_source_dist(data, name="IdeaCatalyst"):
    if type(data) != list:
        # --- Collect source domains ---
        source_domains = []

        for sample in data.values():
            for idx, idea_entry in enumerate(sample.get("proposed_ideas", [])):
                if idx >= 2:
                    break
                domain = idea_entry.get("source_domain")
                if domain is not None:
                    source_domains.append(domain)
                
    else:
        source_domains = data

    # --- Count and sort domains ---
    domain_counts = Counter(source_domains)

    df = (
        pd.DataFrame(domain_counts.items(), columns=["Source Domain", "Count"])
        .sort_values("Count", ascending=False)
    )

    # --- Create bar plot ---
    fig = px.bar(
        df,
        x="Source Domain",
        y="Count",
        color="Source Domain",   # adds color variation
        text="Count",
        title=f"Distribution of Source Domains in {name}"
    )

    # --- Styling for modern look ---
    fig.update_traces(
        textposition="outside"
    )

    fig.update_layout(
        template="seaborn",
        height=600,              # increased height
        font=dict(size=14),
        title_font=dict(size=22),
        xaxis_title_font=dict(size=16),
        yaxis_title_font=dict(size=16),
        bargap=0.3,
        showlegend=False         # cleaner since color = domain
    )

    fig.update_yaxes(
        range=[0, df["Count"].max() * 1.15]  # prevents top cutoff
    )

    fig.show()

In [24]:
prompt = lambda fine_grained_domain : f"""Determine the most relevant **coarse-grained domain** (the target domain which encompasses the subfield, {fine_grained_domain}) that the problem best fits out of the following options (ONLY SELECT THE TARGET DOMAIN FROM THIS LIST):

Computer Science, Medicine, Chemistry, Biology, Materials Science, Physics, Geology, Psychology, Art, History, Geography, Sociology, Business, Political Science, Economics, Philosophy, Mathematics, Engineering, Environmental Science, Agricultural and Food Sciences, Education, Law, Linguistics

Output your answer in JSON format:
{{
    "coarse_grained_domain": [domain from above list]
}}

Subfield: {fine_grained_domain}

Output:

"""

class CoarseDomain(BaseModel):
    coarse_grained_domain: str

In [25]:
def convert_domains(data):
    raw_source_domains = []
    conversion_prompts = []
    for sample in data.values():
        for idea_entry in sample.get("proposed_ideas", []):
            domain = idea_entry.get("source_domain")
            if domain is not None:
                raw_source_domains.append(domain)
                conversion_prompts.append([{"role": "user", "content": prompt(domain)}])
    
    outputs = batch_llm_inference(llm, conversion_prompts, CoarseDomain.model_json_schema(), temperature=0, max_tokens=256)
    return [o["coarse_grained_domain"] for o in outputs]

In [26]:
plot_target_dist(mainmethod_results)

In [19]:
converted_domains = convert_domains(b_two_results)

Processed prompts: 100%|██████████| 1200/1200 [00:10<00:00, 113.07it/s, est. speed input: 15561.84 toks/s, output: 1673.73 toks/s]


In [38]:
plot_source_dist_multi({"Unguided RAG": b_one_results,
                        "Guided RAG": converted_domains,
                        "IdeaCatalyst": mainmethod_results})

In [30]:
plot_source_dist(mainmethod_results)

In [31]:
plot_source_dist(b_one_results, "Unguided Source Retrieval")

In [32]:
plot_source_dist(converted_domains, "Guided Source Retrieval")